## Duplicate Events Detection and Removal

**Information Need:** Identify events that are exact duplicates, and remove them from the event log.

**Motivation:** Exact duplicate events can indicate logging, extraction, or preprocessing errors and may distort event frequencies, activity counts, performance measures, or subsequent process analyses. Once such duplicates have been identified and determined to be unintended, removing them prevents the same recorded event information from being represented multiple times.

**Approach:** Identify events within the same case that have identical values across all event attributes. From each identified group of exact duplicate events, retain a single occurrence and remove the remaining occurrences.

**Output:** The total number of duplicated events in the log, the number of cases where duplicated events occur, and for each case with duplicated events, the case id and the name of the respective activity types. The resulting event log with the identified exact duplicate events removed.

In [ ]:
import pandas as pd
import pm4py

# --- Configuration -----------------------------------------------------------

LOG_PATH = "../../data/Hospital_log.xes"

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
COMPLETION_TIME = "time:timestamp"

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)

display(event_log.head())

## Pattern execution

In [ ]:
# Flag events that have an identical counterpart within the same trace
comparison_columns = [col for col in event_log.columns if col != CASE_ID]

duplicated_mask = event_log.groupby(CASE_ID, sort=False, group_keys=False).apply(
    lambda trace: trace.duplicated(subset=comparison_columns, keep=False)
)

duplicated_events = event_log[duplicated_mask]

num_duplicate_events = len(duplicated_events)
num_affected_cases = duplicated_events[CASE_ID].nunique()

print(f"Duplicate events: {num_duplicate_events}")
print(f"Cases affected by duplicate events: {num_affected_cases}")

In [ ]:
# For each affected case, list the activity names for which a duplicate event exists
duplicate_activities_by_case = (
    duplicated_events.groupby(CASE_ID)[ACTIVITY]
    .apply(lambda names: sorted(set(names)))
    .reset_index(name='duplicated_activities')
)

display(duplicate_activities_by_case)

In [ ]:
# Drop duplicate events within each trace, retaining only the first occurrence
deduplicated_mask = event_log.groupby(CASE_ID, sort=False, group_keys=False).apply(
    lambda trace: ~trace.duplicated(subset=comparison_columns, keep='first')
)

cleaned_event_log = event_log[deduplicated_mask]

print(f"Events before: {len(event_log)}")
print(f"Events after: {len(cleaned_event_log)}")

display(cleaned_event_log.head())